In [4]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.chat_models.tongyi import ChatTongyi
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")

# 加载文档
loader = TextLoader("./knowledge.md",encoding="utf-8")
docs = loader.load()

print(docs)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

splits = text_splitter.split_documents(docs)

print(splits)
embeddings = DashScopeEmbeddings(
    model="text-embedding-v4",
    dashscope_api_key=DASHSCOPE_API_KEY
)

vectorstore = FAISS.from_documents(splits,embeddings)
retriever = vectorstore.as_retriever(search_kwargs = {"k":3})

rag_prompt = ChatPromptTemplate.from_template(
    """
    你是知识库问答助手，**仅能使用下面检索到的上下文回答问题**。

    上下文： {context}
    用户问题：{question}
    """
)

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

# 初始化大模型
model = ChatTongyi(
    model="qwen-long",
    temperature=0,
    dashscope_api_key=DASHSCOPE_API_KEY
)

rag_chain = (
    {"context":retriever | format_docs,"question":RunnablePassthrough()}
    | rag_prompt
    | model
    | StrOutputParser()
)

response = rag_chain.invoke("梅西得到了什么荣誉")
print(response)

[Document(metadata={'source': './knowledge.md'}, page_content='2022年12月18日的卢赛尔球场，35岁的梅西在点球大战后跪倒在草坪中央。阿根廷队以7比5击败法国队，时隔36年重夺世界杯冠军。那场决赛，梅西梅开二度，姆巴佩上演帽子戏法，120分钟战成3比3，最终由点球决出胜负。对梅西而言，这座奖杯补上了他荣誉簿上唯一缺失的那一块；对足球本身而言，它更像是一个迟到的确认——那个被讨论了近二十年“是否史上最佳”的问题，终于有了一个无法被轻易绕过的答案。\n\n**从罗萨里奥到巴塞罗那：一段被生长激素改写的人生**\n\n梅西出生于阿根廷罗萨里奥，少年时代被诊断出生长激素缺乏症。巴塞罗那愿意承担治疗费用，但也因此将他带离了故乡。2004年，他完成一线队首秀，此后十七年间，诺坎普成了他定义足球的地方。他为巴萨赢得35座奖杯，包括10次西甲冠军和4次欧冠冠军。2009年至2012年间，他连续四年捧起金球奖，其中2011-12赛季在西甲打入50球，至今仍是单赛季联赛进球的历史标尺。那段时期的梅西，是一个用频率和重心踢球的人：低到近乎贴着草皮的步频，让他在极小的空间里就能完成变向、加速和射门。你之前拆解过的那个“静止变向”——面对博阿滕，没有踩单车，没有冗余动作，只是两次逼真的内切诱导，然后一次干净的外拨——正是这种能力的浓缩：**他过掉的不是防守球员的身体，而是防守球员的重心决策**。\n\n**巴黎的过渡与卡塔尔的加冕**\n\n2021年离开巴萨时，梅西在告别发布会上落泪。在巴黎圣日耳曼的两个赛季，他收获了2座法甲冠军和1座法国超级杯，但那段经历更像一段必要的过渡，而非新的巅峰。真正的转折发生在国家队。2021年美洲杯，阿根廷击败巴西夺冠，梅西拿到了成年国家队生涯的第一座大赛奖杯。2022年卡塔尔世界杯，他打满7场，贡献7球3助攻，决赛中在加时赛第109分钟补射破门，一度让阿根廷无限接近胜利，直到姆巴佩第117分钟的点球将比赛拖入点球大战。点球大战中，梅西第一个出场命中，达米安·马丁内斯扑出科芒的点球，阿根廷4罚全中，比赛结束。那一刻的梅西没有狂奔，没有嘶吼，只是站在原地，双手微微抬起，像是一个终于确认了某件早已知道的事情的人。\n\n**迈阿密：用头脑替代奔跑的另一种巅峰**\n\n世界杯之后，梅西加盟迈阿密